# v4-1 / v4-2 추천 — models7식 모델별 전처리·학습 (Colab)

REES46로 **다음 행동 추천**. v4 churn 7모델과 같은 프레임의 **다중분류**:
- **v4-1(cat)**: Y=다음 카테고리 / **v4-2(item)**: Y=다음 아이템(top1000)
- 모델: **XGBoost·LightGBM·CatBoost** + **Transformer(tabular)** + **DecisionTree·LogReg·RandomForest**
- 지표: **top-1/top-5/MRR**. 속도: early stopping·hist·작은트리·라벨인코딩·희소클래스제거·층화.

## ★ 결과물: 모델별 전처리 번들
각 모델이 베이지안으로 고른 **최적 전처리(scaler/log/imbalance)+fit된 scaler+모델+클래스맵**을
`prep_<Model>_rec.joblib`(+ `<Model>_rec_train.parquet`·`first30`·`bayes.json`)로 저장하고, **마지막에 zip으로 다운로드**한다.
(= models7의 `prep_<Model>_v2.joblib`와 동일 역할 / 모델별 산출물)

## 입력: 4개 parquet 업로드
`train_cat.parquet, test_cat.parquet, train_item.parquet, test_item.parquet`
> **런타임→GPU** 권장(Transformer 가속).


## 0) 설치


In [ ]:
!pip -q install optuna xgboost lightgbm catboost pyarrow scikit-learn 2>/dev/null
print('installed')


## 1) 임포트 · 디바이스


In [ ]:
import os, json, time, shutil, numpy as np, pandas as pd, joblib
import torch, torch.nn as nn
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEVICE)


## 2) 데이터 업로드 (4개 parquet) + 출력 폴더


In [ ]:
DATA = '/content/recdata'; OUT = '/content/rec_output'; os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
need = ['train_cat.parquet','test_cat.parquet','train_item.parquet','test_item.parquet']
missing = [f for f in need if not os.path.exists(f'{DATA}/{f}')]
if missing:
    try:
        from google.colab import files
        print('아래에서', missing, '업로드:'); up = files.upload()
        for k, v in up.items(): open(f'{DATA}/{os.path.basename(k)}', 'wb').write(v)
    except Exception as e:
        print('업로드 불가 — 드라이브 마운트 후 DATA로 복사하세요:', e)
print('ready:', [f for f in need if os.path.exists(f'{DATA}/{f}')])


## 3) 공통 설정 · 전처리


In [ ]:
FEAT = ['recency_days','tenure_days','ndays','n_events','n_view','n_cart','n_remove_from_cart',
        'n_purchase','avg_price','purch_amt','min_price','max_price','std_price','purchase_avg_price',
        'remove_ratio','cart_purchase_ratio','n_categories','cat_entropy','n_brands','brand_loyalty',
        'n_sessions','events_per_session']
COUNTS = ['ndays','n_events','n_view','n_cart','n_remove_from_cart','n_purchase','purch_amt','n_categories','n_brands','n_sessions']
COUNT_IDX = [FEAT.index(c) for c in COUNTS]
DSETS = {'cat':  dict(train='train_cat.parquet',  test='test_cat.parquet',  y='y_next_category'),
         'item': dict(train='train_item.parquet', test='test_item.parquet', y='y_next_item')}
MIN_COUNT = 10; BOOST = {'XGBoost','LightGBM','CatBoost'}; TREE = BOOST | {'DecisionTree','RandomForest'}

def load_ds(key):
    ds = DSETS[key]
    tr = pd.read_parquet(f'{DATA}/{ds["train"]}'); te = pd.read_parquet(f'{DATA}/{ds["test"]}')
    vc = tr[ds['y']].value_counts(); keep = set(vc[vc >= MIN_COUNT].index)
    tr = tr[tr[ds['y']].isin(keep)]; te = te[te[ds['y']].isin(keep)]
    le = LabelEncoder().fit(tr[ds['y']].values)
    Xtr = np.nan_to_num(tr[FEAT].values.astype(float)); Xte = np.nan_to_num(te[FEAT].values.astype(float))
    ytr = le.transform(tr[ds['y']].values); yte = le.transform(te[ds['y']].values)
    return dict(tr=tr, te=te, Xtr=Xtr, Xte=Xte, ytr=ytr, yte=yte, ncls=len(le.classes_), classes=le.classes_.tolist(), y=ds['y'])

def transform(Xfit, Xapply, prep):
    a, b = Xfit.copy(), Xapply.copy()
    if prep['log_counts']:
        a[:, COUNT_IDX] = np.log1p(np.clip(a[:, COUNT_IDX], 0, None)); b[:, COUNT_IDX] = np.log1p(np.clip(b[:, COUNT_IDX], 0, None))
    sc = {'standard': StandardScaler(), 'minmax': MinMaxScaler(), 'robust': RobustScaler()}.get(prep['scaler'])
    if sc is not None: sc.fit(a); a, b = sc.transform(a), sc.transform(b)
    return a, b, sc          # ★ fit된 scaler도 반환(번들 저장용)

def topk(model, X, y, k=5):
    p = model.predict_proba(X); cls = np.asarray(model.classes_); kk = min(k, p.shape[1])
    idx = np.argpartition(-p, kth=kk-1, axis=1)[:, :kk]; top = cls[idx]
    return float(np.mean([y[i] in top[i] for i in range(len(y))]))

def mrr(model, X, y):
    p = model.predict_proba(X); cls = np.asarray(model.classes_); order = np.argsort(-p, axis=1); rr = 0.0
    for i, yt in enumerate(y):
        r = np.where(cls[order[i]] == yt)[0]
        if len(r): rr += 1.0/(r[0]+1)
    return float(rr/len(y))

def save_bundle(key, name, mtype, prep, hp, scaler, clf, classes, metrics, Xfa, ytr, tr, bp, n_trials):
    mo = f'{OUT}/{key}/{name}'; os.makedirs(mo, exist_ok=True)
    joblib.dump({'model_name': f'{name}_rec_{key}', 'model_type': mtype, 'task':'multiclass_recommendation',
                 'target': DSETS[key]['y'], 'feature_order': FEAT, 'prep': prep, 'hp': hp, 'scaler': scaler,
                 'classifier': clf, 'classes': classes, 'metrics': metrics}, f'{mo}/prep_{name}_rec.joblib')
    pd.DataFrame(Xfa, columns=FEAT).assign(**{DSETS[key]['y']: ytr, 'user_id': tr['user_id'].values}).to_parquet(f'{mo}/{name}_rec_train.parquet', index=False)
    json.dump({'best_params': bp, 'metrics': metrics, 'n_trials': n_trials}, open(f'{mo}/{name}_rec_bayes.json','w',encoding='utf-8'), ensure_ascii=False, indent=2)
    with open(f'{mo}/{name}_first30.txt','w',encoding='utf-8') as f:
        f.write(f'# {name} 추천({key}) — top1 {metrics["oot_top1"]} top5 {metrics["oot_top5"]} MRR {metrics["oot_mrr"]} | prep {prep}\n\n')
        f.write(tr[['user_id']+FEAT+[DSETS[key]['y']]].head(30).to_string(index=False))
    return mo
print('prep ready')


## 4) 부스팅·정형 모델 (베이지안+ES) — 모델별 번들 저장


In [ ]:
N_BOOST = 80; ES = 20; MAX_BIN = 127; USE_GPU_BOOST = (DEVICE == 'cuda')
def make_model(name, hp, cw, ncls):
    if name == 'DecisionTree':
        return DecisionTreeClassifier(max_depth=hp['max_depth'], min_samples_leaf=hp['min_samples_leaf'], class_weight=cw, random_state=SEED)
    if name == 'RandomForest':
        return RandomForestClassifier(n_estimators=hp['n_estimators'], max_depth=hp['max_depth'], min_samples_leaf=hp['min_samples_leaf'], class_weight=cw, n_jobs=-1, random_state=SEED)
    if name == 'LogReg':
        return LogisticRegression(C=hp['C'], class_weight=cw, max_iter=1000, solver='lbfgs', multi_class='multinomial', n_jobs=-1, random_state=SEED)
    if name == 'XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=N_BOOST, max_depth=hp['max_depth'], learning_rate=hp['lr'], subsample=hp['subsample'],
                             colsample_bytree=hp['colsample'], tree_method='hist', max_bin=MAX_BIN, eval_metric='mlogloss',
                             early_stopping_rounds=ES, device=('cuda' if USE_GPU_BOOST else 'cpu'), random_state=SEED, n_jobs=-1)
    if name == 'LightGBM':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=N_BOOST, num_leaves=hp['num_leaves'], learning_rate=hp['lr'], min_child_samples=hp['min_child_samples'],
                              subsample=hp['subsample'], subsample_freq=1, colsample_bytree=hp['colsample'], max_bin=MAX_BIN,
                              class_weight=cw, force_col_wise=True, random_state=SEED, n_jobs=-1, verbose=-1)
    if name == 'CatBoost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(iterations=N_BOOST, depth=hp['depth'], learning_rate=hp['lr'], l2_leaf_reg=hp['l2_leaf_reg'],
                                  loss_function='MultiClass', border_count=MAX_BIN, od_type='Iter', od_wait=ES,
                                  auto_class_weights=('Balanced' if cw else None), task_type=('GPU' if USE_GPU_BOOST else 'CPU'), random_state=SEED, verbose=0)
    raise ValueError(name)

def fit_es(name, model, Xtr, ytr, Xv, yv):
    if name in BOOST:
        seen = np.isin(yv, np.unique(ytr)); Xv, yv = Xv[seen], yv[seen]
    if name == 'XGBoost': model.fit(Xtr, ytr, eval_set=[(Xv, yv)], verbose=False)
    elif name == 'LightGBM':
        import lightgbm as lgb; model.fit(Xtr, ytr, eval_set=[(Xv, yv)], callbacks=[lgb.early_stopping(ES, verbose=False), lgb.log_evaluation(0)])
    elif name == 'CatBoost': model.fit(Xtr, ytr, eval_set=(Xv, yv), verbose=False)
    else: model.fit(Xtr, ytr)
    return model

def sample_hp(t, name):
    if name == 'DecisionTree': return {'max_depth': t.suggest_int('max_depth',4,24), 'min_samples_leaf': t.suggest_int('min_samples_leaf',1,50)}
    if name == 'RandomForest': return {'n_estimators': t.suggest_int('n_estimators',100,300), 'max_depth': t.suggest_int('max_depth',6,28), 'min_samples_leaf': t.suggest_int('min_samples_leaf',1,30)}
    if name == 'LogReg': return {'C': t.suggest_float('C',1e-2,1e2,log=True)}
    if name == 'XGBoost': return {'max_depth': t.suggest_int('max_depth',4,10), 'lr': t.suggest_float('lr',0.03,0.3,log=True), 'subsample': t.suggest_float('subsample',0.7,1.0), 'colsample': t.suggest_float('colsample',0.6,1.0)}
    if name == 'LightGBM': return {'num_leaves': t.suggest_int('num_leaves',15,63), 'lr': t.suggest_float('lr',0.05,0.3,log=True), 'min_child_samples': t.suggest_int('min_child_samples',10,80), 'subsample': t.suggest_float('subsample',0.7,1.0), 'colsample': t.suggest_float('colsample',0.6,1.0)}
    if name == 'CatBoost': return {'depth': t.suggest_int('depth',4,8), 'lr': t.suggest_float('lr',0.05,0.3,log=True), 'l2_leaf_reg': t.suggest_float('l2_leaf_reg',1.0,10.0)}
    return {}

def run_model(key, name, n_trials=3):
    d = load_ds(key); ncls = d['ncls']
    Xh, Xv, yh, yv = train_test_split(d['Xtr'], d['ytr'], test_size=0.2, random_state=SEED, stratify=d['ytr'])
    t0 = time.time()
    def obj(t):
        prep = {'scaler': t.suggest_categorical('scaler', ['none','standard','robust'] if name in TREE else ['standard','minmax','robust']),
                'log_counts': t.suggest_categorical('log_counts', [True, False]), 'imbalance': t.suggest_categorical('imbalance', ['none','classweight'])}
        cw = 'balanced' if prep['imbalance']=='classweight' else None
        Xa, Xb, _ = transform(Xh, Xv, prep)
        m = fit_es(name, make_model(name, sample_hp(t, name), cw, ncls), Xa, yh, Xb, yv)
        return topk(m, Xb, yv, 5)
    st = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED)); st.optimize(obj, n_trials=n_trials)
    bp = st.best_params; prep = {k: bp[k] for k in ('scaler','log_counts','imbalance')}; hp = {k: bp[k] for k in bp if k not in prep}
    cw = 'balanced' if prep['imbalance']=='classweight' else None
    Xfa, Xfb, scaler = transform(d['Xtr'], d['Xte'], prep)
    if name in BOOST:
        Xh2, Xv2, yh2, yv2 = train_test_split(Xfa, d['ytr'], test_size=0.1, random_state=SEED, stratify=d['ytr'])
        clf = fit_es(name, make_model(name, hp, cw, ncls), Xh2, yh2, Xv2, yv2)
    else:
        clf = make_model(name, hp, cw, ncls).fit(Xfa, d['ytr'])
    metrics = dict(oot_top1=round(topk(clf,Xfb,d['yte'],1),4), oot_top5=round(topk(clf,Xfb,d['yte'],5),4),
                   oot_mrr=round(mrr(clf,Xfb,d['yte']),4), n_classes=ncls, n_features=len(FEAT))
    save_bundle(key, name, 'tree' if name in TREE else 'linear', prep, hp, scaler, clf, d['classes'], metrics, Xfa, d['ytr'], d['tr'], bp, n_trials)
    r = dict(dataset=key, model=name, **{k: metrics[k] for k in ('oot_top1','oot_top5','oot_mrr','n_classes')}, prep=f"{prep['scaler']}|{prep['imbalance']}", sec=round(time.time()-t0))
    print(r); return r
print('run_model ready')


## 5) Transformer (tabular) — 모델별 번들 저장


In [ ]:
class TabTransformer(nn.Module):
    def __init__(self, n_feat, n_cls, h=64, heads=4, layers=2, ff=128, drop=0.1):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_feat, h)*0.02); self.femb = nn.Parameter(torch.randn(n_feat, h)*0.02)
        enc = nn.TransformerEncoderLayer(h, heads, ff, batch_first=True, dropout=drop)
        self.tr = nn.TransformerEncoder(enc, layers); self.head = nn.Sequential(nn.LayerNorm(h), nn.Dropout(drop), nn.Linear(h, n_cls))
    def forward(self, x):
        tok = x.unsqueeze(-1)*self.W + self.femb; return self.head(self.tr(tok).mean(1))

def tk_mrr(logits, y, k=5):
    o = np.argsort(-logits, axis=1); t1 = float(np.mean(o[:,0]==y)); tk = float(np.mean([y[i] in o[i,:k] for i in range(len(y))])); rr = 0.0
    for i, yt in enumerate(y):
        r = np.where(o[i]==yt)[0]
        if len(r): rr += 1.0/(r[0]+1)
    return t1, tk, float(rr/len(y))

def run_transformer(key, epochs=40):
    d = load_ds(key); ncls = d['ncls']
    sc = StandardScaler().fit(d['Xtr']); Xtr = sc.transform(d['Xtr']).astype('float32'); Xte = sc.transform(d['Xte']).astype('float32')
    Xh, Xv, yh, yv = train_test_split(Xtr, d['ytr'], test_size=0.2, random_state=SEED, stratify=d['ytr'])
    m = TabTransformer(len(FEAT), ncls).to(DEVICE); opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4); lf = nn.CrossEntropyLoss()
    Xh_t = torch.tensor(Xh, device=DEVICE); yh_t = torch.tensor(yh, dtype=torch.long, device=DEVICE)
    def logits(X):
        m.eval(); out=[]
        with torch.no_grad():
            for i in range(0, len(X), 4096): out.append(m(torch.tensor(X[i:i+4096], device=DEVICE)).cpu().numpy())
        return np.concatenate(out)
    best=-1; best_sd=None; bad=0; t0=time.time()
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(Xh_t), device=DEVICE)
        for i in range(0, len(Xh_t), 512):
            idx = perm[i:i+512]; opt.zero_grad(); loss = lf(m(Xh_t[idx]), yh_t[idx]); loss.backward(); opt.step()
        _, vk, _ = tk_mrr(logits(Xv), yv, 5)
        if vk > best: best, best_sd, bad = vk, {k: v.cpu().clone() for k,v in m.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 6: break
    if best_sd: m.load_state_dict(best_sd)
    t1, t5, mr = tk_mrr(logits(Xte), d['yte'], 5)
    metrics = dict(oot_top1=round(t1,4), oot_top5=round(t5,4), oot_mrr=round(mr,4), n_classes=ncls, n_features=len(FEAT))
    mo = f'{OUT}/{key}/Transformer'; os.makedirs(mo, exist_ok=True)
    joblib.dump({'model_name': f'Transformer_rec_{key}', 'model_type':'transformer_tabular', 'task':'multiclass_recommendation',
                 'target': d['y'], 'feature_order': FEAT, 'scaler': sc, 'state_dict': {k: v.cpu().numpy() for k,v in m.state_dict().items()},
                 'arch': {'h':64,'heads':4,'layers':2,'ff':128}, 'classes': d['classes'], 'metrics': metrics}, f'{mo}/prep_Transformer_rec.joblib')
    pd.DataFrame(Xtr, columns=FEAT).assign(**{d['y']: d['ytr'], 'user_id': d['tr']['user_id'].values}).to_parquet(f'{mo}/Transformer_rec_train.parquet', index=False)
    json.dump({'metrics': metrics, 'epochs_max': epochs, 'arch':{'h':64,'heads':4,'layers':2}}, open(f'{mo}/Transformer_rec_bayes.json','w',encoding='utf-8'), ensure_ascii=False, indent=2)
    r = dict(dataset=key, model='Transformer', oot_top1=round(t1,4), oot_top5=round(t5,4), oot_mrr=round(mr,4), n_classes=ncls, prep='standard', sec=round(time.time()-t0))
    print(r); return r
print('run_transformer ready')


## 6) 전체 실행 (item 부스트 → Transformer → 나머지) — 각 모델 번들 자동 저장


In [ ]:
results = []
for mdl in ['XGBoost','LightGBM','CatBoost']: results.append(run_model('item', mdl, n_trials=3))
results.append(run_transformer('cat')); results.append(run_transformer('item'))
for mdl in ['XGBoost','LightGBM','CatBoost']: results.append(run_model('cat', mdl, n_trials=3))
for key in ['cat','item']:
    for mdl in ['DecisionTree','LogReg','RandomForest']: results.append(run_model(key, mdl, n_trials=4))
R = pd.DataFrame(results); R.to_csv(f'{OUT}/rec_results.csv', index=False)
for key in ['cat','item']:
    print(f'\n=== {key} ==='); print(R[R.dataset==key].sort_values('oot_top5', ascending=False).to_string(index=False))


## 7) ★ 모델별 전처리 산출물 zip 다운로드


In [ ]:
# OUT 구조: rec_output/<cat|item>/<Model>/prep_<Model>_rec.joblib · <Model>_rec_train.parquet · bayes.json · first30.txt
for root,_,fs in os.walk(OUT):
    for f in fs: print(os.path.relpath(os.path.join(root,f), OUT))
zip_path = shutil.make_archive('/content/rec_models7_output', 'zip', OUT)
print('\nZIP:', zip_path, round(os.path.getsize(zip_path)/1e6,1), 'MB')
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print('수동 다운로드: 좌측 파일창에서', zip_path, e)
